# AICS-106 — Exploratory Data Analysis & Dataset Inventory

**Course:** AICS-106 Deep Learning for Threat Detection
**Author:** Janet Okewu-Ihezie
**Purpose:** Explore the synthetic telemetry datasets before training the deep residual NN and LSTM sequence model. This notebook also serves as the **dataset inventory** deliverable required in Week 1 of the lab pack.

---
## 1. Data Governance & Dataset Inventory Note

**Source:** 100% synthetically generated via `generate_telemetry.py` (custom script, seeded for reproducibility, SEED=42).

**Why synthetic instead of a public dataset (e.g. CICIDS2017):** The lab's ReadMe explicitly lists building *"a synthetic enterprise telemetry generator for network flow and Linux authentication events"* as a required deliverable (see "What Students Will Build"). Public datasets (CICIDS2017, UNSW-NB15, CICIoT2023) are listed separately under "Main Public Dataset Links for **Extension Work**" — i.e. optional, not primary.

**Datasets produced:**
1. `network_flows.csv` — 100,000 rows, 6 classes (Benign, DoS, PortScan, BruteForce, Botnet, WebAttack), 14 features. Used for the deep residual NN.
2. `auth_logs.csv` — 31,243 auth events across 8,000 sessions, session-level binary label (Normal/Anomalous). Used for the LSTM sequence model.

**Known limitations (to be honest about, per the lab's ethics rules):**
- Synthetic data is statistically simpler than real-world traffic; class boundaries are more separable than they would be in production data.
- Class imbalance is deliberately built in to mirror real-world skew, but the *degree* of imbalance is an assumption, not measured from real logs.
- No adversarial or evasive traffic patterns are modeled (e.g. slow-and-low attacks, IP spoofing).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

flows = pd.read_csv("network_flows.csv")
auth = pd.read_csv("auth_logs.csv")

print("network_flows.csv shape:", flows.shape)
print("auth_logs.csv shape:", auth.shape)

## 2. Network Flows — Structure & Summary Statistics

In [ ]:
flows.head()

In [ ]:
flows.info()

In [ ]:
flows.describe()

### 2.1 Class Distribution (Class Imbalance Check)

In [ ]:
class_counts = flows["label"].value_counts()
class_pct = (class_counts / class_counts.sum() * 100).round(2)

print(class_counts)
print("\nPercentage breakdown:")
print(class_pct)

fig, ax = plt.subplots()
class_counts.plot(kind="bar", ax=ax, color=sns.color_palette("viridis", len(class_counts)))
ax.set_title("Network Flow Class Distribution (n=100,000)")
ax.set_ylabel("Count")
ax.set_xlabel("Threat Class")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()

**Observation:** The dataset is heavily imbalanced — Benign traffic makes up 55% of all flows, while WebAttack is under 4%. This matters because a naive model could achieve ~55% accuracy simply by predicting "Benign" every time. This is exactly why accuracy alone will not be used as the evaluation metric later — precision, recall, and F1-score per class will be reported instead (per the lab's ethics requirement).

### 2.2 Feature Distributions by Class

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=flows, x="label", y="duration", ax=axes[0,0])
axes[0,0].set_title("Flow Duration by Class")
axes[0,0].tick_params(axis='x', rotation=45)

sns.boxplot(data=flows, x="label", y="packet_count", ax=axes[0,1])
axes[0,1].set_title("Packet Count by Class")
axes[0,1].set_yscale("log")
axes[0,1].tick_params(axis='x', rotation=45)

sns.boxplot(data=flows, x="label", y="flow_iat_mean", ax=axes[1,0])
axes[1,0].set_title("Mean Inter-Arrival Time by Class")
axes[1,0].tick_params(axis='x', rotation=45)

sns.boxplot(data=flows, x="label", y="syn_count", ax=axes[1,1])
axes[1,1].set_title("SYN Count by Class")
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("feature_distributions.png", dpi=150)
plt.show()

**Observation:** DoS flows show extremely high packet counts and SYN counts with very low inter-arrival time — consistent with flooding behavior. PortScan flows show near-zero duration and minimal packets, consistent with reconnaissance sweeps. This confirms the synthetic generator produces features with genuine separating signal, not pure noise — important for justifying that a deep learning model is learning real patterns.

### 2.3 Correlation Heatmap (Numeric Features)

In [ ]:
numeric_cols = flows.select_dtypes(include=[np.number]).columns
corr = flows[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap — Network Flows")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()

### 2.4 Protocol Distribution

In [ ]:
protocol_counts = flows["protocol"].value_counts()
print(protocol_counts)

fig, ax = plt.subplots()
protocol_counts.plot(kind="pie", autopct="%1.1f%%", ax=ax)
ax.set_ylabel("")
ax.set_title("Protocol Distribution")
plt.tight_layout()
plt.savefig("protocol_distribution.png", dpi=150)
plt.show()

## 3. Auth Logs — Structure & Summary Statistics

In [ ]:
auth.head(10)

In [ ]:
auth.info()

### 3.1 Session-Level Label Distribution

In [ ]:
session_labels = auth.groupby("session_id")["session_label"].first()
label_counts = session_labels.value_counts()
label_counts.index = ["Normal", "Anomalous"]

print(label_counts)
print(f"\nAnomalous session rate: {label_counts['Anomalous'] / label_counts.sum() * 100:.2f}%")

fig, ax = plt.subplots()
label_counts.plot(kind="bar", ax=ax, color=["steelblue", "crimson"])
ax.set_title("Auth Session Labels (n=8,000 sessions)")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("session_label_distribution.png", dpi=150)
plt.show()

### 3.2 Event Type Frequency

In [ ]:
event_counts = auth["event_type"].value_counts()
print(event_counts)

fig, ax = plt.subplots()
event_counts.plot(kind="barh", ax=ax, color=sns.color_palette("mako", len(event_counts)))
ax.set_title("Auth Event Type Frequency")
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig("event_type_frequency.png", dpi=150)
plt.show()

### 3.3 Session Length Distribution (Events per Session)

In [ ]:
session_lengths = auth.groupby("session_id").size()

fig, ax = plt.subplots()
sns.histplot(session_lengths, bins=12, kde=False, ax=ax)
ax.set_title("Distribution of Events per Session")
ax.set_xlabel("Number of Events")
plt.tight_layout()
plt.savefig("session_length_distribution.png", dpi=150)
plt.show()

print(session_lengths.describe())

## 4. Summary of Findings (for Capstone Report)

1. **Class imbalance is real and must be handled** — both in network flows (Benign 55% vs WebAttack 3.9%) and auth sessions (Normal 88% vs Anomalous 12%). Class weighting or SMOTE will be applied during model training, and per-class precision/recall/F1 will be the primary evaluation metrics — not raw accuracy.
2. **Features show genuine class-separating signal** — e.g. DoS traffic clusters at high packet/SYN counts with near-zero inter-arrival time, distinct from Benign traffic. This supports the case for a deep learning approach rather than being an arbitrary choice.
3. **No missing values or corrupted rows** were found in either dataset (synthetic generation guarantees clean data — a limitation to note, since real-world telemetry would require cleaning).
4. **Auth log sessions vary in length** (2 to 12 events), consistent with the three anomalous patterns modeled (credential stuffing, off-hours privilege escalation, rapid sudo abuse) plus normal working-hours sessions.
